# REX pre-annotations with LLM

To train or fine-tune a relation extraction model, you need (a lot) of annotated data. To help this process, you can use a large language model (LLM) to generate pre-annotations for your data. This notebook shows you how to do this and how to export the results to a Label Studio project to subsequently review and correct the pre-annotations.


## Step 1: create a prompt for the LLM

The first step in this process is to create a prompt for the LLM. The prompt should contain the following information:
- The task you want the LLM to perform (e.g., relation extraction)
- The schema you want the LLM to use (e.g., the relations you want to extract)
- The type of text you want the LLM to analyze
- The format you want the LLM to use for the output (e.g., JSON).
- if you have prelabeled NER entities, prompt them to be grounded in the text and to use them as entities for relation extraction.

To create a good prompt, I recommend you to use Claude to generate a prompt for you. You can use the following prompt to ask Claude to generate a prompt for you:
```
You are a relation extraction expert. You have to create a prompt for a large language model (LLM) to perform relation extraction on a given text. The prompt should contain the following information:
[insert your version of the above list here].

```
You can find an example of a preannotation prompt in the markdown file ```'pre-annotations/relation_preannotation_prompt.md'```.

NOTE:

if you want to use the evaluation code in the notebook ```NB2-evaluation.ipynb``` and the Label Studio conversion scripts in ```import-export.ipynb```, you need to make sure that the output of the LLM is compatible with the **GLiNER2 output format**:
```
{
  "text": "...",
  "entities": [{"id":"ls-id","start":10,"end":20,"label":"person","text":"..."}],
  "relations": [{"from_id":"ls-id","to_id":"ls-id","label":"is_related_to","direction":"right"}]
}
```
Explicitly mention in your prompt that you want this format. If you don't want to evaluate the pre-annotations and directly export them to Label Studio you can ask the LLM to output the pre-annotations in the **Label Studio format**:
```
{
  "data": {"text": "..."},
  "predictions": [{"result":[{"id":"ls-id","type":"relation","from_name":"relation","to_name":"text","source":"ls-id","target":"ls-id","value":{"relation":"is_related_to"}}]}]
}
```

## Step 2: run the LLM on your data

After creating a prompt, you can feed the prompt to an LLM of your choice (e.g., GPT-4, Claude, etc.) and run it on your data. You can do this in a loop to process multiple texts. The output of the LLM should be in JSON format, which you can then save to a file. I advise you to test a few LLM's to see which one performs best on your data. Keep in mind that most LLM's have a token limit and a limit to the amount of files you can attatch to a prompt. If your data is too large, you can split it into smaller chunks and run the LLM on each chunk separately. Not every LLM gives you the option to download its output, which could slow down the process. Keep that in mind when choosing the LLM.

## Step 3: evaluate the pre-annotations

To know which LLM performs best on your data, you can evaluate the pre-annotations generated by the LLM. You can do this by comparing the pre-annotations to a gold standard (i.e., a set of manually annotated data). To do this, go to the evaluation notebook in this repo called ```NB2-evaluation.ipynb```. In this notebook, you can evaluate the pre-annotations using various metrics (e.g., precision, recall, F1-score). You can also visualize the results using confusion matrices. This will help you to identify which LLM performs best on your data and which relations are more difficult to extract.

## Step 4: export the pre-annotations to Label Studio

After you chose the best performing LLM, you can export the pre-annotations to a Label Studio project as predictions. This will allow you to review and correct the pre-annotations. If you prompted your LLM to output the pre-annotations in the Label Studio format, you can directly import the JSON file into Label Studio. If you prompted your LLM to output the pre-annotations in the GLiNER2 format, you can use the conversion ```8. GLiNER output -> Label Studio PREDICTION JSON``` script in ```import-export.ipynb``` to convert the GLiNER2 output to Label Studio format (or use the same code below).

In [2]:
# if your JSON file(s) is (are) a dictionary instead of a list, use this code to convert it
# if you have a directory full of JSON dictionaries, you can reference the directory, and it will create a list with all the JSON files combined

import os
import json

input_path = "pre-annotations/hagio_preannotation_GPT"

if os.path.isfile(input_path):
    output_path = input_path

    with open(input_path, "r", encoding="utf-8") as f:
        content = f.read()

    wrapped = f'[{content}]' if not content.strip().startswith('[') else content

    with open(output_path, "w", encoding="utf-8") as f:
        f.write(wrapped)

    print(wrapped)

elif os.path.isdir(input_path):
    print("is directory")

    output_path = f'{input_path}/combined/combined.json'

    data = []
    for filename in sorted(os.listdir(input_path)):
        if filename.endswith(".json"):
            with open(os.path.join(input_path, filename), "r", encoding="utf-8") as f:
                file_data = json.load(f)

            if isinstance(file_data, list):
                data.extend(file_data)
            elif isinstance(file_data, dict):
                data.append(file_data)
            else:
                print(f"Skipping {filename}: unexpected top-level type {type(file_data)}")

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    print(f"Saved {len(data)} tasks to {output_path}")

else:
    print("is not a file or directory")

is directory
Saved 38 tasks to pre-annotations/hagio_preannotation_GPT/combined/combined2.json


In [3]:
import json
import uuid
from typing import List, Tuple


def load_json_input(path: str):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def _extract_entities_relations(item: dict) -> Tuple[str, List[dict], List[dict]]:
    """
    Handle two possible input shapes:
      1. Flat GLiNER format: {"text": ..., "entities": [...], "relations": [...]}
      2. LS-shaped input (e.g. someone already ran it through an LS annotation
         step): {"data": {"text": ...}, "annotations": [{"result": [...]}]}
    Returns (text, entities, relations) normalized to the flat GLiNER shape,
    regardless of which form the input came in as.
    """
    if "annotations" in item:
        text = item.get("data", {}).get("text", item.get("text", ""))
        entities, relations = [], []
        anns = item.get("annotations", []) or []
        ann = anns[0] if anns else {}
        for r in ann.get("result", []) or []:
            if r.get("type") == "labels":
                val = r.get("value", {})
                entities.append({
                    "id": r.get("id"),
                    "start": val.get("start"),
                    "end": val.get("end"),
                    "text": val.get("text", ""),
                    "label": (val.get("labels") or [None])[0],
                })
            elif r.get("type") in {"relation", "relations"}:
                relations.append({
                    "from_id": r.get("from_id"),
                    "to_id": r.get("to_id"),
                    "direction": r.get("direction", "right"),
                    "label": (r.get("labels") or [None])[0],
                })
        return text, entities, relations

    text = item.get("text", "")
    entities = item.get("entities", []) or []
    relations = item.get("relations", []) or []
    return text, entities, relations


def gliner_items_to_labelstudio(items: List[dict],
                                model_version: str,
                                from_name: str = "label",
                                to_name: str = "text",
                                result_origin: str = "prediction") -> List[dict]:
    """
    Converts GLiNER-style items (or already LS-shaped items) into Label Studio
    PREDICTION tasks — never annotations — regardless of input shape.
    `model_version` is required and always set by the caller.
    """
    ls_tasks = []
    for i, item in enumerate(items, start=1):
        text, entities, relations = _extract_entities_relations(item)
        results = []

        # Maps that let a relation resolve its from_id/to_id no matter which
        # convention the source file used to reference entities:
        #   - integer / digit-string -> positional index into `entities`
        #   - the entity's own original "id" string (if it had one)
        index_to_ls_id = {}
        original_id_to_ls_id = {}

        for idx, ent in enumerate(entities):
            start, end = ent.get("start"), ent.get("end")
            if start is None or end is None or ent.get("label") is None:
                continue  # skip malformed spans rather than crash

            # Preserve the original entity id when the source provided one,
            # so relations referencing that id still resolve correctly.
            # Fall back to a generated id only if none was given.
            orig_id = ent.get("id")
            ls_id = str(orig_id) if orig_id else str(uuid.uuid4())[:10]

            index_to_ls_id[idx] = ls_id
            if orig_id:
                original_id_to_ls_id[str(orig_id)] = ls_id

            results.append({
                "id": ls_id,
                "from_name": from_name,
                "to_name": to_name,
                "type": "labels",
                "origin": result_origin,
                "value": {
                    "start": start,
                    "end": end,
                    "text": ent.get("text", text[start:end]),
                    "labels": [ent["label"]],
                }
            })

        def _resolve_id(ref):
            if ref is None:
                return None
            if isinstance(ref, int) or (isinstance(ref, str) and ref.isdigit()):
                return index_to_ls_id.get(int(ref))
            # try matching against the original entity id string
            return original_id_to_ls_id.get(str(ref), ref)

        for rel in relations:
            if rel.get("label") is None:
                continue
            from_id = _resolve_id(rel.get("from_id"))
            to_id = _resolve_id(rel.get("to_id"))
            if from_id is None or to_id is None:
                continue  # skip relations we couldn't resolve rather than crash
            results.append({
                "id": str(uuid.uuid4())[:10],
                "type": "relation",
                "origin": result_origin,
                "from_id": from_id,
                "to_id": to_id,
                "direction": rel.get("direction", "right"),
                "labels": [rel["label"]],
            })

        ls_tasks.append({
            "id": i,
            "data": {"text": text},
            "predictions": [{
                "model_version": model_version,
                "result": results
            }]
        })
    return ls_tasks


# Example usage
from pathlib import Path

input_path = "pre-annotations/hagio_preannotation_GPT/combined/combined2.json"
output_path = str(Path(input_path).with_name(Path(input_path).stem + "_LS2.json"))

gliner_items = load_json_input(input_path)
ls_from_gliner = gliner_items_to_labelstudio(gliner_items, model_version="REX-annotation")

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(ls_from_gliner, f, ensure_ascii=False, indent=2)

print(f"Saved {len(ls_from_gliner)} Label Studio prediction tasks to {output_path}")
if ls_from_gliner:
    print(ls_from_gliner[0])

Saved 38 Label Studio prediction tasks to pre-annotations\hagio_preannotation_GPT\combined\combined2_LS2.json
{'id': 1, 'data': {'text': '[5] Dicitur tamen hæc sancta virgo Glodesinda fuisse temporibus Childerici regis, & cujusdam illustris ducis filia, qui dux Wintro vocabatur. Matris quoque ejus nomen Godila erat; atque satis prædicta proles nobili generis stemmate procreata. Quæ cum in tantum jam fuisset adulta, ut sibi conjugii copula provideri deberet; in ea efficientia secundum filiorum generositatem ipse pater laborare inde jam cœperat; resque usque ad desponsionem pervenit. Peracto vero sponsionis tempore, accipiens eam pater ejus materque, sive parentes nobiles, prædictam Christi virginem ac sponsam Domini nostri Jesu Christi Glodesindam, tradiderunt eam cuidam viro nobili, Oboleno nomine; qui accipiens eam, duxit ad regendam domum suam: introductamque prædictam virginem ac sponsam Domini nostri Jesu Christi Glodesindam intra septa domus suæ, volens copulari ei, attamen Domini